# Partie 2 — Traitement batch avec SparkCore (RDD)
Prérequis : `trips_history.jsonl` est dans MinIO : `lake/bronze/history/`.

In [1]:
from common import get_spark
import json, time

spark = get_spark("Pedali-Part2-RDD")
sc = spark.sparkContext
print(spark.version, "| cœurs :", sc.defaultParallelism)

3.5.3 | cœurs : 2


## 1. Chargement et parsing (transformation lazy + 1 action)

In [2]:
PATH = "s3a://lake/bronze/history/trips_history.jsonl"

raw = sc.textFile(PATH)
print("Lignes brutes :", raw.count())

def parse(line):
    try:
        return json.loads(line)
    except Exception:
        return None

trips = raw.map(parse).filter(lambda t: t is not None)

Lignes brutes : 50000


## 2. Dédoublonnage (~2 % de doublons injectés : même trip_id + même status)

In [3]:
n_avant = trips.count()

trips_dedup = (trips
    .map(lambda t: ((t["trip_id"], t["status"]), t))
    .reduceByKey(lambda a, b: a)
    .map(lambda kv: kv[1]))

n_apres = trips_dedup.count()
print(f"Avant : {n_avant} | Après : {n_apres} | Doublons retirés : {n_avant - n_apres}")

Avant : 50000 | Après : 49040 | Doublons retirés : 960


## 3. Filtre : trajets terminés (ENDED) + cache

In [4]:
ended = trips_dedup.filter(lambda t: t["status"] == "ENDED").cache()
print("Trajets ENDED :", ended.count())

Trajets ENDED : 15612


## 4. Chiffre d'affaires par ville (map + reduceByKey)

In [5]:
t0 = time.time()
ca_ville = (ended
    .map(lambda t: (t["city"], t["price_mad"] or 0.0))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda kv: -kv[1])
    .collect())
t_ca = time.time() - t0

print(f"Temps : {t_ca:.2f}s\n")
print(f"{'Ville':<14}{'CA total (MAD)':>16}")
for ville, ca in ca_ville:
    print(f"{ville:<14}{ca:>16,.2f}")

Temps : 2.97s

Ville           CA total (MAD)
Casablanca          111,883.46
Marrakech            58,128.99
Rabat                56,013.93
Tanger               45,385.42


## 5. Durée moyenne par ville et type de véhicule (somme + nombre en un seul passage)

In [6]:
t0 = time.time()
zero = (0, 0)
seq  = lambda acc, d: (acc[0] + d, acc[1] + 1)
comb = lambda a, b: (a[0] + b[0], a[1] + b[1])

duree = (ended
    .map(lambda t: ((t["city"], t["vehicle_type"]), t["duration_s"] or 0))
    .aggregateByKey(zero, seq, comb)
    .mapValues(lambda s: (s[0] / s[1] / 60, s[1]))
    .sortByKey()
    .collect())
t_duree = time.time() - t0

print(f"Temps : {t_duree:.2f}s\n")
print(f"{'Ville':<14}{'Véhicule':<10}{'Durée moy (min)':>17}{'Nb trajets':>12}")
for (ville, veh), (moy, n) in duree:
    print(f"{ville:<14}{veh:<10}{moy:>17.1f}{n:>12}")

Temps : 3.65s

Ville         Véhicule    Durée moy (min)  Nb trajets
Casablanca    BIKE                   27.9        1758
Casablanca    EBIKE                  21.2        2351
Casablanca    SCOOTER                22.9        1734
Marrakech     BIKE                   25.0        1100
Marrakech     EBIKE                  17.5        1479
Marrakech     SCOOTER                17.9         989
Rabat         BIKE                   23.5         999
Rabat         EBIKE                  17.5        1346
Rabat         SCOOTER                18.9        1033
Tanger        BIKE                   23.7         850
Tanger        EBIKE                  16.8        1131
Tanger        SCOOTER                17.9         842


## 6. Top 5 des stations de départ

In [7]:
t0 = time.time()
top5 = (ended
    .map(lambda t: (t["start_station"], 1))
    .reduceByKey(lambda a, b: a + b)
    .takeOrdered(5, key=lambda kv: -kv[1]))
t_top5 = time.time() - t0

print(f"Temps : {t_top5:.2f}s")
for i, (st, n) in enumerate(top5, 1):
    print(f"{i}. {st}  ->  {n} trajets")

Temps : 1.30s
1. S-MAR-002  ->  166 trajets
2. S-MAR-004  ->  156 trajets
3. S-MAR-005  ->  150 trajets
4. S-CAS-001  ->  144 trajets
5. S-MAR-001  ->  134 trajets


## 7. Lineage (toDebugString)
Transformations (lazy) : textFile, map, filter, reduceByKey, aggregateByKey, mapValues, sortBy, sortByKey, cache.
Actions : count, collect, takeOrdered, sum.

In [8]:
rdd_final = (ended
    .map(lambda t: (t["city"], t["price_mad"] or 0.0))
    .reduceByKey(lambda a, b: a + b))

ds = rdd_final.toDebugString()
print(ds.decode("utf-8") if isinstance(ds, bytes) else ds)

(2) PythonRDD[42] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[41] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[40] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[39] at reduceByKey at /tmp/ipykernel_992/1087112924.py:3 []
    |  PythonRDD[38] at reduceByKey at /tmp/ipykernel_992/1087112924.py:3 []
    |  PythonRDD[9] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 1106.0 KiB; DiskSize: 0.0 B
    |  MapPartitionsRDD[7] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[6] at partitionBy at NativeMethodAccessorImpl.java:0 []
    +-(2) PairwiseRDD[5] at reduceByKey at /tmp/ipykernel_992/1377985638.py:5 []
       |  PythonRDD[4] at reduceByKey at /tmp/ipykernel_992/1377985638.py:5 []
       |  s3a://lake/bronze/history/trips_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
       |  s3a://lake/bronze/history/trips_history.jsonl HadoopRDD[0] at textFile at NativeMethodAc

## 8. Récapitulatif des temps (à comparer avec la Partie 3)

In [9]:
print("=== Temps RDD ===")
print(f"CA par ville   : {t_ca:.2f}s")
print(f"Durée moyenne  : {t_duree:.2f}s")
print(f"Top 5 stations : {t_top5:.2f}s")

=== Temps RDD ===
CA par ville   : 2.97s
Durée moyenne  : 3.65s
Top 5 stations : 1.30s


📸 Captures : Spark UI http://localhost:4040 → Jobs → job du CA → *DAG Visualization* ; onglet *Stages* (shuffle read/write) ; onglet *Storage*.
Notez les résultats (CA, durées, top 5) : la Partie 3 doit retrouver **exactement** les mêmes.
Puis **Kernel → Shut Down Kernel**.